# Autism Spectrum Disorder (ASD) Prediction

## Project Information
- **Data Source**: Autism Prediction Dataset (Kaggle)
- **Objective**: Predict Autism Spectrum Disorder (ASD) based on behavioral and demographic features
- **Models Used**: SVC, Logistic Regression, LightGBM, XGBoost

## Dataset
The dataset contains:
- A1-A10 Scores: Behavioral screening questionnaire scores
- Demographic information (age, gender, ethnicity, country of residence)
- Medical history (jaundice, autism in family)
- Application usage history
- Target: Class/ASD (0 = No ASD, 1 = ASD)


In [ ]:
%pip install -r ../requirements.txt


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)


In [ ]:
# Load and explore the dataset
df = pd.read_csv('autism_prediction.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nTarget distribution:\n{df['Class/ASD'].value_counts()}")
df.head()


## Data Preprocessing


In [ ]:
# Drop unnecessary columns
drop_cols = ['age_desc', 'ID']
df = df.drop(columns=drop_cols)

# Replace categorical values
df = df.replace({'yes': 1, 'no': 0, '?': 'others', 'Others': 'others'})

# Handle country of residence: keep top 10, rest as 'others'
top_10_countries = df['contry_of_res'].value_counts().head(10).index.to_list()
df.loc[~df['contry_of_res'].isin(top_10_countries), 'contry_of_res'] = 'others'

# Remove missing values
df.dropna(inplace=True)
print(f"Dataset shape after preprocessing: {df.shape}")
print(f"Target distribution:\n{df['Class/ASD'].value_counts()}")


In [ ]:
# Define column types for preprocessing
int_cols = [f"A{i}_Score" for i in range(1, 11)]
float_cols = ['age', 'result']
bin_cols = ['jaundice', 'austim', 'used_app_before'] + int_cols
cat_cols = ['gender', 'ethnicity', 'contry_of_res', 'relation']
num_cols = bin_cols + float_cols

# Prepare features and target
X = df.drop(columns=['Class/ASD'])
y = df['Class/ASD'].values

print(f"Features shape: {X.shape}")
print(f"Continuous columns: {float_cols}")
print(f"Binary columns: {len(bin_cols)}")
print(f"Categorical columns: {cat_cols}")


In [ ]:
# Split data into train and test sets -- make sure to stratify the target
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train set: {X_train_raw.shape}")
print(f"Test set: {X_test.shape}")


## Preprocessing Pipeline

We'll use different preprocessing for tree-based models vs. linear models:
- **For linear models (SVC, Logistic Regression)**: Power transform + Standard scaling for continuous features
- **For tree-based models (LightGBM, XGBoost)**: Only one-hot encode categorical features


In [ ]:
# Preprocessing for linear models (SVC, Logistic Regression)
num_transformer = Pipeline([
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler())
])

cat_transformer = OneHotEncoder(handle_unknown='ignore')

linear_preprocess = ColumnTransformer([
    ('floats', num_transformer, float_cols),
    ('bins', 'passthrough', bin_cols),
    ('cat', cat_transformer, cat_cols)
])

# Preprocessing for tree-based models
tree_preprocess = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Apply preprocessing
X_train_processed = linear_preprocess.fit_transform(X_train_raw)
X_test_processed = linear_preprocess.transform(X_test)

# Handle class imbalance with SMOTE
X_train, y_train = SMOTE(random_state=SEED).fit_resample(X_train_processed, y_train_raw)
print(f"After SMOTE - Train set: {X_train.shape}, Target distribution: {np.bincount(y_train)}")


## Model 1: Support Vector Classifier (SVC)


In [ ]:
# Train SVC model
svc_model = SVC(kernel='rbf', probability=True, random_state=SEED)
svc_model.fit(X_train, y_train)

# Make predictions
svc_preds = svc_model.predict(X_test_processed)
svc_proba = svc_model.predict_proba(X_test_processed)[:, 1]

print("SVC Results:")
print(classification_report(y_test, svc_preds))
print(f"Accuracy: {accuracy_score(y_test, svc_preds):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, svc_proba):.4f}")


## Model 2: Logistic Regression


In [ ]:
# Train Logistic Regression with balanced class weights
log_reg = Pipeline([
    ('preprocess', linear_preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))
])

log_reg.fit(X_train_raw, y_train_raw)

# Make predictions
lr_preds = log_reg.predict(X_test)
lr_proba = log_reg.predict_proba(X_test)[:, 1]

print("Logistic Regression Results:")
print(classification_report(y_test, lr_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")


## Model 3: LightGBM


In [ ]:
# Prepare data for tree-based models
X_train_tree = tree_preprocess.fit_transform(X_train_raw)
X_test_tree = tree_preprocess.transform(X_test)

# Train LightGBM
lgb_model = Pipeline([
    ('preprocess', tree_preprocess),
    ('model', lgb.LGBMClassifier(
        max_depth=3, 
        num_leaves=35, 
        random_state=SEED, 
        is_unbalance=True,
        verbose=-1
    ))
])

lgb_model.fit(X_train_raw, y_train_raw)

# Make predictions
lgb_preds = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]

print("LightGBM Results:")
print(classification_report(y_test, lgb_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, lgb_proba):.4f}")


## Model 4: XGBoost


In [ ]:
# Calculate scale_pos_weight for imbalanced data
scale_pos_weight = (y_train_raw == 0).sum() / (y_train_raw == 1).sum()

# Train XGBoost
xgb_model = Pipeline([
    ('preprocess', tree_preprocess),
    ('model', xgb.XGBClassifier(
        random_state=SEED,
        max_depth=5,
        learning_rate=0.05,
        n_estimators=500,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss'
    ))
])

xgb_model.fit(X_train_raw, y_train_raw)

# Make predictions
xgb_preds = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Results:")
print(classification_report(y_test, xgb_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_proba):.4f}")


## ROC Curve Visualization


In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 8))

models = {
    'SVC': svc_proba,
    'Logistic Regression': lr_proba,
    'LightGBM': lgb_proba,
    'XGBoost': xgb_proba
}

for name, y_proba in models.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()
